# 10 - Perbandingan 4-Class dan 7-Class

**Tujuan:** Membandingkan performa semua model pada konfigurasi 4-class, serta membandingkan hasil 4-class vs 7-class untuk menentukan apakah penggabungan kelas minoritas meningkatkan performa.

**4 Kelas Emosi:** neutral, happy, sad, negative (gabungan angry+fearful+disgusted+surprised)

**7 Kelas Emosi:** neutral, happy, sad, angry, fearful, disgusted, surprised

**3 Skenario Imbalance:**
- B1: Tanpa penanganan (baseline)
- B2: Dengan class weights
- B3: Dengan class weights + augmentasi

## 1. Setup

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Project root
PROJECT_ROOT = Path("..").resolve()

# Plot style
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12
sns.set_style("whitegrid")

SCENARIOS = ["B1", "B2", "B3"]
EMOTIONS_4 = ["neutral", "happy", "sad", "negative"]
EMOTIONS_7 = ["neutral", "happy", "sad", "angry", "fearful", "disgusted", "surprised"]

print(f"Project root: {PROJECT_ROOT}")

## 2. Load Results

In [ ]:
# --- Load 4-class results ---
model_dirs_4class = {
    "CNN": PROJECT_ROOT / "models" / "4class" / "cnn" / "cnn_4class_results.json",
    "FCNN": PROJECT_ROOT / "models" / "4class" / "fcnn" / "fcnn_4class_results.json",
    "Late Fusion": PROJECT_ROOT / "models" / "4class" / "late_fusion" / "late_fusion_4class_results.json",
    "Intermediate Fusion": PROJECT_ROOT / "models" / "4class" / "intermediate_fusion" / "intermediate_fusion_4class_results.json",
}

results_4class = {}
for model_name, json_path in model_dirs_4class.items():
    if json_path.exists():
        with open(json_path, "r") as f:
            results_4class[model_name] = json.load(f)
        print(f"[OK] 4-class {model_name} loaded from {json_path.name}")
    else:
        print(f"[WARN] File not found: {json_path}")

print(f"\nTotal 4-class models loaded: {len(results_4class)}")

In [ ]:
# --- Load 7-class results ---
model_dirs_7class = {
    "CNN": PROJECT_ROOT / "models" / "cnn" / "cnn_results.json",
    "FCNN": PROJECT_ROOT / "models" / "fcnn" / "fcnn_results.json",
    "Late Fusion": PROJECT_ROOT / "models" / "late_fusion" / "late_fusion_results.json",
    "Intermediate Fusion": PROJECT_ROOT / "models" / "intermediate_fusion" / "intermediate_fusion_results.json",
}

results_7class = {}
for model_name, json_path in model_dirs_7class.items():
    if json_path.exists():
        with open(json_path, "r") as f:
            results_7class[model_name] = json.load(f)
        print(f"[OK] 7-class {model_name} loaded from {json_path.name}")
    else:
        print(f"[WARN] File not found: {json_path}")

print(f"\nTotal 7-class models loaded: {len(results_7class)}")

In [ ]:
def extract_metrics(model_results, scenario_key):
    """Extract accuracy, macro_f1, weighted_f1 from a scenario result dict.
    
    Handles different key naming conventions across model types:
    - CNN/FCNN: 'accuracy', 'macro_f1', 'weighted_f1'
    - Late/Intermediate Fusion: 'test_accuracy', 'test_macro_f1', 'test_weighted_f1'
    """
    metrics = model_results[scenario_key]
    accuracy = metrics.get("accuracy", metrics.get("test_accuracy", 0))
    macro_f1 = metrics.get("macro_f1", metrics.get("test_macro_f1", 0))
    weighted_f1 = metrics.get("weighted_f1", metrics.get("test_weighted_f1", 0))
    return accuracy, macro_f1, weighted_f1


def find_scenario_key(model_results, scenario_short):
    """Find the actual key in the results dict that matches B1/B2/B3.
    
    Handles keys like 'B1', 'B1 Baseline', 'B2 Class Weights', etc.
    """
    for key in model_results.keys():
        if key.startswith(scenario_short):
            return key
    return None


print("Helper functions defined.")

## 3. 4-Class Results Summary

Tabel lengkap performa 4 model x 3 skenario pada konfigurasi 4-class.

In [ ]:
# Build summary table for 4-class results
rows_4class = []
for model_name, scenarios in results_4class.items():
    for scenario_short in SCENARIOS:
        scenario_key = find_scenario_key(scenarios, scenario_short)
        if scenario_key is None:
            print(f"[WARN] {model_name}: scenario {scenario_short} not found")
            continue
        accuracy, macro_f1, weighted_f1 = extract_metrics(scenarios, scenario_key)
        rows_4class.append({
            "Model": model_name,
            "Skenario": scenario_short,
            "Accuracy": accuracy,
            "Macro F1": macro_f1,
            "Weighted F1": weighted_f1,
        })

df_4class = pd.DataFrame(rows_4class)
print(f"Total rows: {len(df_4class)} (expected: {len(results_4class)} models x 3 scenarios = {len(results_4class)*3})")
print()

styled_4class = df_4class.style \
    .format({"Accuracy": "{:.4f}", "Macro F1": "{:.4f}", "Weighted F1": "{:.4f}"}) \
    .background_gradient(subset=["Macro F1"], cmap="YlGn") \
    .set_caption("4-Class Results: Semua Model dan Skenario")
styled_4class

In [ ]:
# Best 4-class combination
best_4class_idx = df_4class["Macro F1"].idxmax()
best_4class = df_4class.loc[best_4class_idx]

print("=" * 60)
print("KOMBINASI TERBAIK 4-CLASS (berdasarkan Macro F1)")
print("=" * 60)
print(f"  Model       : {best_4class['Model']}")
print(f"  Skenario    : {best_4class['Skenario']}")
print(f"  Accuracy    : {best_4class['Accuracy']:.4f}")
print(f"  Macro F1    : {best_4class['Macro F1']:.4f}")
print(f"  Weighted F1 : {best_4class['Weighted F1']:.4f}")
print("=" * 60)

print("\nRanking semua kombinasi 4-class:")
ranking_4class = df_4class.sort_values("Macro F1", ascending=False).reset_index(drop=True)
ranking_4class.index = ranking_4class.index + 1
ranking_4class.index.name = "Rank"
display(ranking_4class)

## 4. 4-Class vs 7-Class Comparison

Perbandingan side-by-side: hasil terbaik per model pada konfigurasi 4-class vs 7-class.
Menunjukkan apakah penggabungan kelas minoritas (angry, fearful, disgusted, surprised) ke "negative" meningkatkan performa.

In [ ]:
# Build 7-class summary table
rows_7class = []
for model_name, scenarios in results_7class.items():
    for scenario_short in SCENARIOS:
        scenario_key = find_scenario_key(scenarios, scenario_short)
        if scenario_key is None:
            continue
        accuracy, macro_f1, weighted_f1 = extract_metrics(scenarios, scenario_key)
        rows_7class.append({
            "Model": model_name,
            "Skenario": scenario_short,
            "Accuracy": accuracy,
            "Macro F1": macro_f1,
            "Weighted F1": weighted_f1,
        })

df_7class = pd.DataFrame(rows_7class)

# Get best result per model for each class config
best_per_model_4 = df_4class.loc[df_4class.groupby("Model")["Macro F1"].idxmax()].reset_index(drop=True)
best_per_model_7 = df_7class.loc[df_7class.groupby("Model")["Macro F1"].idxmax()].reset_index(drop=True)

# Merge into side-by-side comparison
comparison = best_per_model_7.merge(
    best_per_model_4,
    on="Model",
    suffixes=("_7class", "_4class")
)

# Calculate improvement
comparison["Macro F1 Delta"] = comparison["Macro F1_4class"] - comparison["Macro F1_7class"]
comparison["Accuracy Delta"] = comparison["Accuracy_4class"] - comparison["Accuracy_7class"]
comparison["Macro F1 Improvement %"] = (
    (comparison["Macro F1_4class"] - comparison["Macro F1_7class"]) / comparison["Macro F1_7class"] * 100
)

# Display
display_cols = [
    "Model",
    "Skenario_7class", "Macro F1_7class", "Accuracy_7class",
    "Skenario_4class", "Macro F1_4class", "Accuracy_4class",
    "Macro F1 Delta", "Macro F1 Improvement %",
]

styled_comparison = comparison[display_cols].style \
    .format({
        "Macro F1_7class": "{:.4f}",
        "Accuracy_7class": "{:.4f}",
        "Macro F1_4class": "{:.4f}",
        "Accuracy_4class": "{:.4f}",
        "Macro F1 Delta": "{:+.4f}",
        "Macro F1 Improvement %": "{:+.1f}%",
    }) \
    .background_gradient(subset=["Macro F1 Delta"], cmap="RdYlGn") \
    .set_caption("4-Class vs 7-Class: Best Result per Model")

styled_comparison

In [ ]:
# Print summary of improvements
print("=" * 70)
print("RINGKASAN PERBANDINGAN 4-CLASS vs 7-CLASS")
print("=" * 70)

for _, row in comparison.iterrows():
    direction = "IMPROVEMENT" if row["Macro F1 Delta"] > 0 else "DEGRADATION"
    print(f"\n--- {row['Model']} ---")
    print(f"  7-class best: {row['Skenario_7class']} -> Macro F1 = {row['Macro F1_7class']:.4f}")
    print(f"  4-class best: {row['Skenario_4class']} -> Macro F1 = {row['Macro F1_4class']:.4f}")
    print(f"  Delta: {row['Macro F1 Delta']:+.4f} ({row['Macro F1 Improvement %']:+.1f}%) [{direction}]")

avg_improvement = comparison["Macro F1 Improvement %"].mean()
print(f"\n{'=' * 70}")
print(f"Rata-rata perubahan Macro F1: {avg_improvement:+.1f}%")
print(f"{'=' * 70}")

## 5. Visualization

Grouped bar chart membandingkan Macro F1 terbaik per model antara 7-class dan 4-class.

In [ ]:
# Grouped bar chart: 7-class vs 4-class Macro F1 per model (best scenario each)
fig, ax = plt.subplots(figsize=(12, 7))

models = comparison["Model"].tolist()
f1_7class = comparison["Macro F1_7class"].tolist()
f1_4class = comparison["Macro F1_4class"].tolist()

x = np.arange(len(models))
width = 0.35

bars_7 = ax.bar(x - width/2, f1_7class, width, label="7-Class", color="#3498db", edgecolor="white", linewidth=0.5)
bars_4 = ax.bar(x + width/2, f1_4class, width, label="4-Class", color="#e74c3c", edgecolor="white", linewidth=0.5)

# Add value labels
for bar in bars_7:
    height = bar.get_height()
    ax.annotate(f"{height:.4f}",
                xy=(bar.get_x() + bar.get_width() / 2, height),
                xytext=(0, 4), textcoords="offset points",
                ha="center", va="bottom", fontsize=9, fontweight="bold")

for bar in bars_4:
    height = bar.get_height()
    ax.annotate(f"{height:.4f}",
                xy=(bar.get_x() + bar.get_width() / 2, height),
                xytext=(0, 4), textcoords="offset points",
                ha="center", va="bottom", fontsize=9, fontweight="bold")

# Add improvement arrows/annotations
for i, (_, row) in enumerate(comparison.iterrows()):
    delta = row["Macro F1 Delta"]
    color = "green" if delta > 0 else "red"
    ax.annotate(f"{delta:+.4f}",
                xy=(i, max(f1_7class[i], f1_4class[i]) + 0.02),
                ha="center", fontsize=9, color=color, fontweight="bold")

ax.set_xlabel("Model", fontsize=13)
ax.set_ylabel("Macro F1 Score", fontsize=13)
ax.set_title("Perbandingan Macro F1: 7-Class vs 4-Class (Best Scenario per Model)",
             fontsize=14, fontweight="bold")
ax.set_xticks(x)
ax.set_xticklabels(models, fontsize=11)
ax.legend(title="Konfigurasi Kelas", fontsize=11)
ax.set_ylim(0, max(max(f1_7class), max(f1_4class)) + 0.10)
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig(PROJECT_ROOT / "models" / "4class" / "comparison_4class_vs_7class.png",
            dpi=150, bbox_inches="tight")
plt.show()
print("Saved: models/4class/comparison_4class_vs_7class.png")

In [ ]:
# Additional visualization: 4-class models across all scenarios
fig, ax = plt.subplots(figsize=(14, 7))

pivot_4class = df_4class.pivot(index="Model", columns="Skenario", values="Macro F1")
col_order = [s for s in SCENARIOS if s in pivot_4class.columns]
pivot_4class = pivot_4class[col_order]

x = np.arange(len(pivot_4class.index))
width = 0.25
colors = ["#3498db", "#e74c3c", "#2ecc71"]

for i, scenario in enumerate(col_order):
    bars = ax.bar(x + i * width, pivot_4class[scenario], width,
                  label=scenario, color=colors[i], edgecolor="white", linewidth=0.5)
    for bar in bars:
        height = bar.get_height()
        ax.annotate(f"{height:.3f}",
                    xy=(bar.get_x() + bar.get_width() / 2, height),
                    xytext=(0, 4), textcoords="offset points",
                    ha="center", va="bottom", fontsize=9, fontweight="bold")

ax.set_xlabel("Model", fontsize=13)
ax.set_ylabel("Macro F1 Score", fontsize=13)
ax.set_title("4-Class: Macro F1 per Model dan Skenario", fontsize=15, fontweight="bold")
ax.set_xticks(x + width)
ax.set_xticklabels(pivot_4class.index, fontsize=11)
ax.legend(title="Skenario", fontsize=11)
ax.set_ylim(0, min(1.0, pivot_4class.max().max() + 0.15))
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig(PROJECT_ROOT / "models" / "4class" / "comparison_4class_scenarios.png",
            dpi=150, bbox_inches="tight")
plt.show()
print("Saved: models/4class/comparison_4class_scenarios.png")

## 6. Per-Class Analysis

Analisis per-kelas untuk model 4-class terbaik: kelas mana yang paling meningkat dibandingkan 7-class.

In [ ]:
# Find the best 4-class model and load its per-class metrics
best_model_name = best_4class["Model"]
best_scenario = best_4class["Skenario"]

print(f"Best 4-class model: {best_model_name} ({best_scenario})")
print()

# Load per-class report from the best 4-class model results
best_4class_data = results_4class[best_model_name]
best_scenario_key = find_scenario_key(best_4class_data, best_scenario)
best_scenario_data = best_4class_data[best_scenario_key]

# Check if per-class report is available
per_class_key = None
for key in ["classification_report", "per_class", "class_report", "test_classification_report"]:
    if key in best_scenario_data:
        per_class_key = key
        break

if per_class_key:
    report = best_scenario_data[per_class_key]
    print(f"Per-class report found (key: '{per_class_key}')")
    print()
    
    # Build per-class dataframe
    per_class_rows = []
    for emotion in EMOTIONS_4:
        if emotion in report:
            metrics = report[emotion]
            per_class_rows.append({
                "Class": emotion,
                "Precision": metrics.get("precision", 0),
                "Recall": metrics.get("recall", 0),
                "F1-Score": metrics.get("f1-score", 0),
                "Support": metrics.get("support", 0),
            })
    
    if per_class_rows:
        df_perclass = pd.DataFrame(per_class_rows)
        styled_perclass = df_perclass.style \
            .format({"Precision": "{:.4f}", "Recall": "{:.4f}", "F1-Score": "{:.4f}", "Support": "{:.0f}"}) \
            .background_gradient(subset=["F1-Score"], cmap="YlGn") \
            .set_caption(f"Per-Class Metrics: {best_model_name} ({best_scenario}) - 4-Class")
        display(styled_perclass)
        
        # Visualize per-class F1
        fig, ax = plt.subplots(figsize=(10, 6))
        colors_bar = ["#2ecc71" if v >= 0.3 else "#e74c3c" for v in df_perclass["F1-Score"]]
        bars = ax.bar(df_perclass["Class"], df_perclass["F1-Score"], color=colors_bar, edgecolor="white")
        for bar in bars:
            height = bar.get_height()
            ax.annotate(f"{height:.4f}",
                        xy=(bar.get_x() + bar.get_width() / 2, height),
                        xytext=(0, 4), textcoords="offset points",
                        ha="center", va="bottom", fontsize=10, fontweight="bold")
        ax.set_ylabel("F1-Score", fontsize=13)
        ax.set_title(f"Per-Class F1-Score: {best_model_name} ({best_scenario}) - 4-Class",
                     fontsize=14, fontweight="bold")
        ax.set_ylim(0, 1.0)
        ax.grid(axis="y", alpha=0.3)
        plt.tight_layout()
        plt.savefig(PROJECT_ROOT / "models" / "4class" / "best_model_perclass_f1.png",
                    dpi=150, bbox_inches="tight")
        plt.show()
    else:
        print("No per-class data found for the expected emotion labels.")
else:
    print("Per-class classification report not available in the results JSON.")
    print(f"Available keys: {list(best_scenario_data.keys())}")
    print()
    print("Note: Per-class analysis requires the classification_report to be saved")
    print("in the results JSON. You may need to re-run the training notebook with")
    print("per-class metrics saving enabled.")

In [ ]:
# Compare shared classes between 7-class and 4-class for the best model
# Shared classes: neutral, happy, sad (present in both configs)
# 4-class has 'negative' which merges angry+fearful+disgusted+surprised from 7-class

print("=" * 60)
print("PERBANDINGAN KELAS BERSAMA (neutral, happy, sad)")
print(f"Model: {best_model_name}")
print("=" * 60)

# Try to get 7-class per-class report for comparison
best_7class_data = results_7class.get(best_model_name, {})
best_7class_scenario_key = None
best_7class_f1 = -1
for scenario_short in SCENARIOS:
    sk = find_scenario_key(best_7class_data, scenario_short)
    if sk:
        _, mf1, _ = extract_metrics(best_7class_data, sk)
        if mf1 > best_7class_f1:
            best_7class_f1 = mf1
            best_7class_scenario_key = sk

if best_7class_scenario_key:
    data_7 = best_7class_data[best_7class_scenario_key]
    report_7 = None
    for key in ["classification_report", "per_class", "class_report", "test_classification_report"]:
        if key in data_7:
            report_7 = data_7[key]
            break
    
    if report_7 and per_class_key:
        report_4 = best_scenario_data[per_class_key]
        shared_classes = ["neutral", "happy", "sad"]
        
        compare_rows = []
        for cls in shared_classes:
            f1_7 = report_7.get(cls, {}).get("f1-score", 0)
            f1_4 = report_4.get(cls, {}).get("f1-score", 0)
            compare_rows.append({
                "Class": cls,
                "F1 (7-class)": f1_7,
                "F1 (4-class)": f1_4,
                "Delta": f1_4 - f1_7,
            })
        
        # Add negative vs merged minority classes
        minority_classes = ["angry", "fearful", "disgusted", "surprised"]
        minority_f1s = [report_7.get(cls, {}).get("f1-score", 0) for cls in minority_classes]
        avg_minority_f1 = np.mean(minority_f1s) if minority_f1s else 0
        negative_f1 = report_4.get("negative", {}).get("f1-score", 0)
        compare_rows.append({
            "Class": "negative (vs avg minority)",
            "F1 (7-class)": avg_minority_f1,
            "F1 (4-class)": negative_f1,
            "Delta": negative_f1 - avg_minority_f1,
        })
        
        df_compare = pd.DataFrame(compare_rows)
        styled_cmp = df_compare.style \
            .format({"F1 (7-class)": "{:.4f}", "F1 (4-class)": "{:.4f}", "Delta": "{:+.4f}"}) \
            .background_gradient(subset=["Delta"], cmap="RdYlGn") \
            .set_caption(f"Per-Class F1 Comparison: {best_model_name}")
        display(df_compare)
    else:
        print("Per-class reports not available for detailed comparison.")
        print("Overall comparison (model-level) is shown in Section 4 above.")
else:
    print("Could not load 7-class results for this model.")

## 7. Kesimpulan

### Temuan Utama

Berdasarkan perbandingan eksperimen 4-class vs 7-class:

1. **Performa 4-Class Terbaik:** _(diisi setelah eksperimen)_
   - Model: ...
   - Skenario: ...
   - Macro F1: ...

2. **Perbandingan dengan 7-Class:**
   - Rata-rata peningkatan/penurunan Macro F1: ...
   - Apakah penggabungan kelas minoritas ke "negative" efektif? ...

3. **Analisis per Kelas:**
   - Kelas yang paling meningkat: ...
   - Kelas "negative" (gabungan) vs kelas minoritas terpisah: ...

4. **Pengaruh Penanganan Imbalance pada 4-Class:**
   - Apakah B2/B3 masih diperlukan ketika kelas sudah digabung? ...
   - Skenario mana yang optimal untuk 4-class? ...

5. **Rekomendasi:**
   - Konfigurasi kelas yang direkomendasikan (4 vs 7): ...
   - Model dan skenario yang direkomendasikan: ...

### Catatan
- 4-class mapping: angry, fearful, disgusted, surprised -> "negative"
- Metric utama: **Macro F1** (memberikan bobot yang sama pada setiap kelas)
- Semua evaluasi dilakukan pada test set yang sama

## 8. Save Combined Results

In [ ]:
import os

# Build combined summary
summary_output = {
    "experiment": "Multimodal Emotion Recognition - 4-Class vs 7-Class Comparison",
    "emotions_4class": EMOTIONS_4,
    "emotions_7class": EMOTIONS_7,
    "models": list(results_4class.keys()),
    "scenarios": SCENARIOS,
    "results_4class": [],
    "results_7class": [],
    "best_4class": {
        "model": best_4class["Model"],
        "scenario": best_4class["Skenario"],
        "accuracy": float(best_4class["Accuracy"]),
        "macro_f1": float(best_4class["Macro F1"]),
        "weighted_f1": float(best_4class["Weighted F1"]),
    },
    "comparison": [],
}

# Add 4-class results
for _, row in df_4class.iterrows():
    summary_output["results_4class"].append({
        "model": row["Model"],
        "scenario": row["Skenario"],
        "accuracy": float(row["Accuracy"]),
        "macro_f1": float(row["Macro F1"]),
        "weighted_f1": float(row["Weighted F1"]),
    })

# Add 7-class results
for _, row in df_7class.iterrows():
    summary_output["results_7class"].append({
        "model": row["Model"],
        "scenario": row["Skenario"],
        "accuracy": float(row["Accuracy"]),
        "macro_f1": float(row["Macro F1"]),
        "weighted_f1": float(row["Weighted F1"]),
    })

# Add comparison (best per model)
for _, row in comparison.iterrows():
    summary_output["comparison"].append({
        "model": row["Model"],
        "best_7class_scenario": row["Skenario_7class"],
        "best_7class_macro_f1": float(row["Macro F1_7class"]),
        "best_4class_scenario": row["Skenario_4class"],
        "best_4class_macro_f1": float(row["Macro F1_4class"]),
        "macro_f1_delta": float(row["Macro F1 Delta"]),
        "improvement_pct": float(row["Macro F1 Improvement %"]),
    })

# Sort results by macro_f1 descending
summary_output["results_4class"].sort(key=lambda x: x["macro_f1"], reverse=True)
summary_output["results_7class"].sort(key=lambda x: x["macro_f1"], reverse=True)

# Save
output_path = PROJECT_ROOT / "models" / "4class" / "experiment_summary_4class.json"
os.makedirs(output_path.parent, exist_ok=True)

with open(output_path, "w") as f:
    json.dump(summary_output, f, indent=2, ensure_ascii=False)

print(f"Summary saved to: {output_path}")
print(f"4-class combinations: {len(summary_output['results_4class'])}")
print(f"7-class combinations: {len(summary_output['results_7class'])}")
print(f"Model comparisons: {len(summary_output['comparison'])}")
print()
print("Best 4-class combination:")
print(f"  {summary_output['best_4class']['model']} + {summary_output['best_4class']['scenario']}")
print(f"  Macro F1: {summary_output['best_4class']['macro_f1']:.4f}")